# Scholarix Data Trust Dashboard - Starting EDA

In [1]:
from collections import Counter
from html import escape
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = str
    display = print

cwd = Path.cwd()
audit_dir = cwd / "analysis" / "audit" if (cwd / "analysis" / "audit").exists() else cwd
if str(audit_dir) not in sys.path:
    sys.path.insert(0, str(audit_dir))

import external_checks
import fetch_data
import internal_checks

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

In [2]:
internal_data = internal_checks.get_internal_data()
external_data = fetch_data.fetch_all()
external_comparisons = external_checks.compare_against_external_data(internal_data, external_data)

print(f"Authors loaded: {len(internal_data)}")
print(f"Unique DOIs checked: {len(external_data['dois'])}")

Fetching
OpenAlex authors: 50/50
OpenAlex publications by author: 50/50
ORCID records: 28/28
OpenAlex publications by DOI: 4624/4624
Crossref publications: 4624/4624
DataCite publications: 67/67
DOI resolutions: 9/9
Semantic Scholar publications by DOI: 4624/4624
Authors loaded: 50
Unique DOIs checked: 4624


## 1. Data Quality Scorecard

This scorecard summarizes which fields appear more trustworthy and which fields create product risk.

In [3]:
def pct(part, total):
    return round(100 * part / total, 1) if total else None


def agreement_percent(matches):
    total = sum(matches.values())
    agreed = matches.get("exact", 0) + matches.get("normalized_match", 0)
    return pct(agreed, total)


def year_agreement_percent(matches):
    total = sum(matches.values())
    return pct(matches.get(0, 0), total)


reference = external_comparisons["reference_agreement"]
across_sources = reference["across sources"]
availability = external_checks.summarize_doi_availability(external_data)
duplicate_summary = internal_checks.summarize_duplicate_dois(internal_data)
orcid_names = pd.DataFrame(external_comparisons["orcid_names"])
orcid_recoverability = pd.DataFrame(external_comparisons["orcid_recoverability"])
s2_evidence = pd.DataFrame(external_comparisons["semantic_scholar_author_evidence"])
profile_verification = pd.DataFrame(internal_checks.compare_profile_verifications(internal_data))
markup = internal_checks.get_titles_with_markup(internal_data)

registered_dois = availability.get("crossref", 0) + availability.get("datacite_not_crossref", 0)
orcid_mismatches = 0 if orcid_names.empty else int((orcid_names["status"] == "mismatch").sum())
recoverable_orcids = int((orcid_recoverability["status"] == "recoverable").sum())
s2_id_counts = s2_evidence.groupby("author_id")["semantic_scholar_author_id"].nunique() if not s2_evidence.empty else pd.Series(dtype="int64")
merged_candidates = int((s2_id_counts >= 5).sum())
profile_link_failures = profile_verification.loc[
    profile_verification["verification_status"].isin(["missing", "unverified"])
].shape[0]
title_markup_count = sum(len(items) for items in markup.values())

scorecard = pd.DataFrame(
    [
        {
            "area": "DOI availability",
            "trust_level": "High",
            "evidence": f"{registered_dois}/{len(external_data['dois'])} DOIs found in Crossref or DataCite ({pct(registered_dois, len(external_data['dois']))}%).",
            "product_meaning": "DOIs are a strong anchor for validation and deduplication.",
        },
        {
            "area": "Title metadata",
            "trust_level": "High",
            "evidence": f"{agreement_percent(across_sources['title_matches'])}% title agreement across reference sources.",
            "product_meaning": "Titles are usually reliable enough to show, but formatting cleanup is still needed.",
        },
        {
            "area": "Publication year",
            "trust_level": "High/Medium",
            "evidence": f"{year_agreement_percent(across_sources['year_gaps'])}% exact year agreement across reference sources.",
            "product_meaning": "Years are mostly stable, but year disagreements should be explainable.",
        },
        {
            "area": "Journal / venue",
            "trust_level": "Medium/Low",
            "evidence": f"{agreement_percent(across_sources['journal_matches'])}% journal agreement across reference sources.",
            "product_meaning": "Venue data needs confidence labels because sources disagree or omit values.",
        },
        {
            "area": "Duplicate publications",
            "trust_level": "Medium/Low",
            "evidence": f"{duplicate_summary['duplicate_dois']} duplicate DOI groups found within author publication lists.",
            "product_meaning": "Duplicate records can inflate publication counts and confuse users.",
        },
        {
            "area": "ORCID identity",
            "trust_level": "Low/Medium",
            "evidence": f"{orcid_mismatches} ORCID name mismatches; {recoverable_orcids} missing ORCIDs appear recoverable from OpenAlex.",
            "product_meaning": "Identity fields need manual review before users trust profiles.",
        },
        {
            "area": "Author disambiguation",
            "trust_level": "Risky",
            "evidence": f"{merged_candidates} profiles match 5+ Semantic Scholar author IDs.",
            "product_meaning": "Some profiles may combine evidence from multiple real people.",
        },
        {
            "area": "Profile link verification",
            "trust_level": "Low",
            "evidence": f"{profile_link_failures} profile-link checks are missing or unverified.",
            "product_meaning": "Users need source transparency instead of a simple verified/unverified flag.",
        },
        {
            "area": "Title cleanliness",
            "trust_level": "Medium",
            "evidence": f"{title_markup_count} titles contain HTML or MathML-like markup.",
            "product_meaning": "Some records need cleanup before they look credible in a UI.",
        },
    ]
)

display(scorecard)

,area,trust_level,evidence,product_meaning
0,DOI availability,High,4615/4624 DOIs found in Crossref or DataCite (99.8%).,DOIs are a strong anchor for validation and deduplication.
1,Title metadata,High,99.5% title agreement across reference sources.,"Titles are usually reliable enough to show, but formatting cleanup is still needed."
2,Publication year,High/Medium,92.3% exact year agreement across reference sources.,"Years are mostly stable, but year disagreements should be explainable."
3,Journal / venue,Medium/Low,84.4% journal agreement across reference sources.,Venue data needs confidence labels because sources disagree or omit values.
4,Duplicate publications,Medium/Low,481 duplicate DOI groups found within author publication lists.,Duplicate records can inflate publication counts and confuse users.
5,ORCID identity,Low/Medium,9 ORCID name mismatches; 19 missing ORCIDs appear recoverable from OpenAlex.,Identity fields need manual review before users trust profiles.
6,Author disambiguation,Risky,27 profiles match 5+ Semantic Scholar author IDs.,Some profiles may combine evidence from multiple real people.
7,Profile link verification,Low,150 profile-link checks are missing or unverified.,Users need source transparency instead of a simple verified/unverified flag.
8,Title cleanliness,Medium,416 titles contain HTML or MathML-like markup.,Some records need cleanup before they look credible in a UI.


## 2. Author Risk Heatmap

Each row is an author. Each issue column is a flag or count. This helps identify which profiles should be reviewed first.

In [4]:
author_names = {
    author_id: author["profile"].get("name")
    for author_id, author in internal_data.items()
}

agreement_by_author = pd.DataFrame(external_checks.agreement_by_author(internal_data, external_data))
journal_agreement = agreement_by_author.set_index("author_id")["journal_agreement"].to_dict()

orcid_status = orcid_names.set_index("author_id")["status"].to_dict() if not orcid_names.empty else {}
recoverability_status = orcid_recoverability.set_index("author_id")["status"].to_dict()

duplicate_dois_by_author = Counter()
title_markup_by_author = Counter()
unresolved_dois_by_author = Counter()

crossref_dois = set(external_data["crossref_publications"])
datacite_dois = set(external_data["datacite_publications"])
redirected_dois = {
    doi for doi, resolution in external_data["doi_resolutions"].items()
    if resolution.get("redirected")
}

for author_id, author in internal_data.items():
    records_by_doi = {}
    for publication in author["publications"]:
        doi = fetch_data.normalize_doi(publication.get("doi"))
        if doi:
            records_by_doi.setdefault(doi, []).append(publication)
            if doi not in crossref_dois and doi not in datacite_dois and doi not in redirected_dois:
                unresolved_dois_by_author[author_id] += 1

        title = publication.get("title", "")
        if internal_checks.MARKUP_PATTERNS["mml"].search(title) or internal_checks.MARKUP_PATTERNS["html"].search(title):
            title_markup_by_author[author_id] += 1

    duplicate_dois_by_author[author_id] = sum(1 for records in records_by_doi.values() if len(records) > 1)

profile_verification_pivot = profile_verification.pivot_table(
    index="author_id",
    columns="source",
    values="verification_status",
    aggfunc="first",
)

risk_rows = []
for author_id in internal_data:
    row = {
        "author_id": author_id,
        "profile_name": author_names[author_id],
        "orcid_mismatch": int(orcid_status.get(author_id) == "mismatch"),
        "recoverable_orcid": int(recoverability_status.get(author_id) == "recoverable"),
        "low_journal_agreement": int((journal_agreement.get(author_id) or 100) < 75),
        "semantic_scholar_ids": int(s2_id_counts.get(author_id, 0)),
        "possible_merged_profile": int(s2_id_counts.get(author_id, 0) >= 5),
        "duplicate_doi_groups": int(duplicate_dois_by_author[author_id]),
        "title_markup_records": int(title_markup_by_author[author_id]),
        "unresolved_doi_records": int(unresolved_dois_by_author[author_id]),
        "google_scholar_unverified": int(
            profile_verification_pivot.get("google_scholar", pd.Series()).get(author_id) == "unverified"
        ),
    }
    row["risk_score"] = (
        3 * row["orcid_mismatch"]
        + row["recoverable_orcid"]
        + 2 * row["low_journal_agreement"]
        + 3 * row["possible_merged_profile"]
        + 2 * int(row["duplicate_doi_groups"] > 0)
        + int(row["title_markup_records"] > 0)
        + int(row["unresolved_doi_records"] > 0)
        + row["google_scholar_unverified"]
    )
    risk_rows.append(row)

risk_df = pd.DataFrame(risk_rows).sort_values(
    ["risk_score", "orcid_mismatch", "possible_merged_profile", "duplicate_doi_groups"],
    ascending=False,
)

heatmap_columns = [
    "profile_name",
    "risk_score",
    "orcid_mismatch",
    "recoverable_orcid",
    "low_journal_agreement",
    "possible_merged_profile",
    "semantic_scholar_ids",
    "duplicate_doi_groups",
    "title_markup_records",
    "unresolved_doi_records",
    "google_scholar_unverified",
]

def issue_style(value):
    if value == 0:
        return "background:#f8fafc;color:#475569"
    if value == 1:
        return "background:#fee2e2;color:#991b1b"
    if value < 5:
        return "background:#fecaca;color:#7f1d1d"
    return "background:#ef4444;color:white"


def render_heatmap(df):
    visible = df[heatmap_columns].head(25)
    header = "".join(f"<th>{escape(column)}</th>" for column in visible.columns)
    rows = []
    for record in visible.to_dict("records"):
        cells = []
        for column in visible.columns:
            value = record[column]
            if column == "profile_name":
                style = "background:#fff;color:#111827;text-align:left"
            else:
                style = issue_style(value)
            cells.append(f"<td style='{style}'>{escape(str(value))}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    return HTML(
        "<table style='border-collapse:collapse;font-family:Arial,sans-serif;font-size:12px'>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody>"
        "</table>"
    )


display(render_heatmap(risk_df))

profile_name,risk_score,orcid_mismatch,recoverable_orcid,low_journal_agreement,possible_merged_profile,semantic_scholar_ids,duplicate_doi_groups,title_markup_records,unresolved_doi_records,google_scholar_unverified
Philip S. Yu,11,1,0,1,1,7,19,0,0,1
I. Petrov,10,1,0,0,1,6,15,9,0,1
Stephen P. Long,10,1,0,0,1,7,7,26,0,1
Boxuan Zhao,10,1,0,0,1,15,6,4,0,1
Marshall Scott Poole,10,1,0,1,0,3,8,2,1,1
Steven J. Clough,10,0,1,1,1,6,8,25,0,1
Tarek Abdelzaher,9,1,0,1,0,4,11,0,1,1
Michael A. Peters,9,0,0,1,1,6,15,1,0,1
Brian D. Fields,9,0,1,0,1,7,11,10,1,1
Bruce Hannon,9,0,0,1,1,5,10,0,1,1


## 3. Source Agreement Charts

These simple bar charts compare source agreement. They are intentionally plain so the story is easy to read.

In [5]:
def make_percent_rows(counter, label, order=None):
    total = sum(counter.values())
    keys = order or sorted(counter, key=lambda key: str(key))
    return [
        {"metric": label, "status": str(key), "records": counter.get(key, 0), "percent": pct(counter.get(key, 0), total)}
        for key in keys
        if key in counter
    ]


chart_rows = []
for source_side, summary in reference.items():
    chart_rows.extend(make_percent_rows(summary["title_matches"], f"Title - {source_side}", ["exact", "normalized_match", "mismatch", "unavailable"]))
    chart_rows.extend(make_percent_rows(summary["journal_matches"], f"Journal - {source_side}", ["exact", "normalized_match", "mismatch", "unavailable"]))
    chart_rows.extend(make_percent_rows(summary["year_gaps"], f"Year gap - {source_side}", [0, 1, 2, 3, 5, 10, None]))

doi_rows = make_percent_rows(availability, "DOI availability", ["crossref", "datacite_not_crossref", "resolves_only", "unresolved"])
chart_rows.extend(doi_rows)
charts_df = pd.DataFrame(chart_rows)


def render_bars(df):
    sections = []
    for metric, metric_df in df.groupby("metric", sort=False):
        rows = []
        for record in metric_df.to_dict("records"):
            width = max(record["percent"] or 0, 0)
            rows.append(
                "<tr>"
                f"<td style='width:190px'>{escape(record['status'])}</td>"
                f"<td style='width:90px;text-align:right'>{record['records']}</td>"
                f"<td style='width:70px;text-align:right'>{record['percent']}%</td>"
                "<td style='width:360px'>"
                f"<div style='height:14px;width:{width * 3}px;max-width:300px;background:#2563eb;border-radius:3px'></div>"
                "</td>"
                "</tr>"
            )
        sections.append(
            f"<h4 style='margin:18px 0 6px'>{escape(metric)}</h4>"
            "<table style='border-collapse:collapse;font-family:Arial,sans-serif;font-size:13px'>"
            + "".join(rows)
            + "</table>"
        )
    return HTML("".join(sections))


display(render_bars(charts_df))

exact,4607,99.3%,
normalized_match,18,0.4%,
mismatch,14,0.3%,
unavailable,1,0.0%,
exact,4248,91.6%,
normalized_match,9,0.2%,
mismatch,17,0.4%,
unavailable,366,7.9%,
0,4637,99.9%,
1,1,0.0%,
None,2,0.0%,


## 4. Review Queue Prototype

This table is a first prototype of the workflow the product could support: show the highest-risk profiles first, explain the flags, and suggest the next review action.

In [6]:
orcid_details = orcid_names.set_index("author_id").to_dict("index") if not orcid_names.empty else {}


def risk_tier(score):
    if score >= 7:
        return "High risk"
    if score >= 3:
        return "Needs review"
    return "Monitor"


def review_evidence(row):
    evidence = []
    author_id = row["author_id"]
    if row["orcid_mismatch"]:
        details = orcid_details.get(author_id, {})
        evidence.append(
            f"ORCID name mismatch: profile='{details.get('profile_name')}', ORCID='{details.get('orcid_name')}'"
        )
    if row["possible_merged_profile"]:
        evidence.append(f"Matched {row['semantic_scholar_ids']} Semantic Scholar author IDs")
    if row["low_journal_agreement"]:
        evidence.append(f"Low journal agreement: {journal_agreement.get(author_id)}%")
    if row["duplicate_doi_groups"]:
        evidence.append(f"{row['duplicate_doi_groups']} duplicate DOI groups")
    if row["recoverable_orcid"]:
        evidence.append("Internal ORCID missing, but external source may have one")
    if row["title_markup_records"]:
        evidence.append(f"{row['title_markup_records']} titles contain markup")
    if row["unresolved_doi_records"]:
        evidence.append(f"{row['unresolved_doi_records']} DOI records unresolved")
    if row["google_scholar_unverified"]:
        evidence.append("Google Scholar profile verification is unverified")
    return "; ".join(evidence[:4])


def suggested_action(row):
    if row["orcid_mismatch"]:
        return "Review identity before showing ORCID as trusted"
    if row["possible_merged_profile"]:
        return "Check whether profile merges multiple researchers"
    if row["duplicate_doi_groups"]:
        return "Deduplicate publications before using counts"
    if row["low_journal_agreement"]:
        return "Show venue confidence/source disagreement"
    return "Monitor; no major identity flag found"


queue = risk_df.copy()
queue["risk_tier"] = queue["risk_score"].map(risk_tier)
queue["why_flagged"] = queue.apply(review_evidence, axis=1)
queue["suggested_action"] = queue.apply(suggested_action, axis=1)

review_queue = queue[
    [
        "risk_tier",
        "risk_score",
        "profile_name",
        "author_id",
        "why_flagged",
        "suggested_action",
    ]
].sort_values(["risk_score", "risk_tier"], ascending=False)

display(review_queue.head(30))

,risk_tier,risk_score,profile_name,author_id,why_flagged,suggested_action
38,High risk,11,Philip S. Yu,A5036357902,"ORCID name mismatch: profile='Philip S. Yu', ORCID='Xuan Lin'; Matched 7 Semantic Scholar author IDs; Low journal ag...",Review identity before showing ORCID as trusted
19,High risk,10,I. Petrov,A5010177657,"ORCID name mismatch: profile='I. Petrov', ORCID='Mikhail I. Petrov'; Matched 6 Semantic Scholar author IDs; 15 dupli...",Review identity before showing ORCID as trusted
44,High risk,10,Stephen P. Long,A5074295593,"ORCID name mismatch: profile='Stephen P. Long', ORCID='Rachel G Shekar'; Matched 7 Semantic Scholar author IDs; 7 du...",Review identity before showing ORCID as trusted
5,High risk,10,Boxuan Zhao,A5003524720,"ORCID name mismatch: profile='Boxuan Zhao', ORCID='Boxuan Simen Zhao'; Matched 15 Semantic Scholar author IDs; 6 dup...",Review identity before showing ORCID as trusted
32,High risk,10,Marshall Scott Poole,A5034256615,"ORCID name mismatch: profile='Marshall Scott Poole', ORCID='Marja Turunen'; Low journal agreement: 58.4%; 8 duplicat...",Review identity before showing ORCID as trusted
45,High risk,10,Steven J. Clough,A5038539249,"Matched 6 Semantic Scholar author IDs; Low journal agreement: 73.1%; 8 duplicate DOI groups; Internal ORCID missing,...",Check whether profile merges multiple researchers
46,High risk,9,Tarek Abdelzaher,A5087114395,"ORCID name mismatch: profile='Tarek Abdelzaher', ORCID='Tarek F. Abdelzaher'; Low journal agreement: 27.1%; 11 dupli...",Review identity before showing ORCID as trusted
34,High risk,9,Michael A. Peters,A5090163591,Matched 6 Semantic Scholar author IDs; Low journal agreement: 72.0%; 15 duplicate DOI groups; 1 titles contain markup,Check whether profile merges multiple researchers
6,High risk,9,Brian D. Fields,A5013197591,"Matched 7 Semantic Scholar author IDs; 11 duplicate DOI groups; Internal ORCID missing, but external source may have...",Check whether profile merges multiple researchers
7,High risk,9,Bruce Hannon,A5030681852,Matched 5 Semantic Scholar author IDs; Low journal agreement: 68.2%; 10 duplicate DOI groups; 1 DOI records unresolved,Check whether profile merges multiple researchers
